In [258]:
'''
Talina Daragmeh 1634835
Tony Trinh 1606724
Timur Sultanov 1614939
'''

'\nTalina Daragmeh 1634835\nTony Trinh 1606724\nTimur Sultanov 1614939\n'

In [259]:
import csv
from collections import defaultdict
import pandas as pd

In [260]:
'''s1: getting surname of a person to determine the blocks'''
def get_surname(name):
    parts = name.strip().split()
    if not parts:
        return None
    return parts[-1].upper()

In [261]:
'''s2: getting initials of a person to determine the blocks'''
def get_initials(name):
    parts = name.strip().split()
    if not parts:
        return None
    firstname_initial = parts[0][0].upper() if parts[0] else None
    surname_initial = parts[-1][0].upper()  if parts[-1] else None

    if firstname_initial is None or surname_initial is None:
        return None
    
    return firstname_initial + surname_initial

In [262]:
'''getting blocks, based on blocking function'''
def compute_blocks(all_names, blocking_function):
    blocks = defaultdict(list)
    
    for name in all_names:
        key = blocking_function(name)
        if key:
            blocks[key].append(name)
    return blocks

In [263]:
'''
P: set of all name pairs
PO ⊆ P set of pairs, that end up in the same block 
Rec := |PO| / |P|
'''
def calculate_rec(pairs, blocking_function):
    '''how many of the pairs are in the same block?'''
    P = len(pairs)
    PO = 0
    for old_name, new_name in pairs:
        old_key = blocking_function(old_name)
        new_key = blocking_function(new_name)
        
        if old_key == new_key:
            PO += 1
        
    return PO / P 

In [264]:
'''
C: sum of the number of size two subsets in all blocks (the actual
number of similarities that need to be computed after the blocking) -> sum(nb * (nb-1)/2)
nb: number of mentions in a block 
n: number of all mentions -> denominator save: n*(n-1)/2 
B: set of all blocks 
'''
def calculate_save(blocks, n):
    c = 0
    for names in blocks.values():
        nb = len(names) 
        c += nb*(nb-1)/2

    total_pairs_possible = n*(n-1)/2 
    return 1 - (c/total_pairs_possible)

In [265]:
path = "dblp_names.csv"
pairs = []
all_names = set() #mentions -> unique

with open(path, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader: 
        #only rows with exactly two columns included (assignment defines each line as a pair of OLD_NAME and NEW_NAME) -> rows with different number of columns excluded
        if len(row) != 2:
            continue
        old_name = row[0].strip()
        new_name = row[1].strip()
        
        pairs.append((old_name, new_name))
        all_names.add(old_name)
        all_names.add(new_name)
        
    blocks_s1 = compute_blocks(all_names, get_surname)
    blocks_s2 = compute_blocks(all_names, get_initials)
        
    rec_s1 = calculate_rec(pairs, get_surname)
    rec_s2 = calculate_rec(pairs, get_initials)
    
    n = len(all_names)
    save_s1 = calculate_save(blocks_s1,n)
    save_s2 = calculate_save(blocks_s2,n)

In [266]:
results = pd.DataFrame({
    "Strategy": ["s1 surname", "s2 initials"],
    "Rec": [rec_s1, rec_s2],
    "Save": [save_s1, save_s2]
})
results

,Strategy,Rec,Save
0,s1 surname,0.770019,0.999633
1,s2 initials,0.906590,0.996715


In [267]:
# Compare those values and briefly comment on the differences that you observe.
# Provide a short explanation why the results differ. Which strategy would you prefer?
"""
REC: how many of the pairs are really in the same block? -> value should be as high as possible 
SAVE: how many comparisons are saved by using blocking? -> value should be as high as possible

bigger blocks: rec higher, save lower 
smaller blocks: rec lower, save higher

Strategy s1 uses surnames as block key. The blocks are more specific and the number of required comparisons is overall pretty low for each block. However, matching pairs can be missed, if there are differences in the spelling of the surname or if the name is incomplete. That is the reason,  the rec value is not as high as in Strategy s2. 

Strategy s2 uses the initials of the first name and the surname as a two-letter key. Therefore, far fewer but larger blocks are created, which increases the chance that matching pairs share the same block.

Rec calculates how many of the pairs are grouped together in the same block. As the probability is higher for s2 the matching pairs to be in the same block, the rec score is higher. However more comparisons are necessary, as the save score is a little bit lower. Save describes the saving if this block is used. s1's Save is a little bit higher (99.96 % vs. 99.67 %). The reason is that s1 produces many more, smaller blocks (one per distinct surname), reducing the number of within-block comparisons. s2 produces blocks based on two-letter initials, meaning each block is much larger and contains far more name pairs to compare. Nevertheless, the difference in save is very small, so the gain in recall far outweighs the small loss in Save. 
------------------------ 
"""
# Name at least two properties of the underlying name base that might affect the usefulness of the strategies outlines above.
"""
1. Frequency distribution of surnames/initials: If many persons share the same surname, the corresponding blocks in s1 become very large, increasing C and reducing Save. For s2 this effect is even stronger, since fewer possible two-letter combinations exist, which means that common initials can cover thousands of persons, resulting in very large blocks with many unnecessary comparisons.

2. Special Characters: Names containing Umlaute or special characters produce different block keys, even though they refer to the same person. s1 is directly affected by this, since even a minor difference in the surname spelling results in different blocks and the pair is permanently missed. s2 is less affected, as long as the first character of the surname remains the same, the pair still lands in the same block and is not lost. But if the first character is affected, s2 encounters the same problem. 
"""

'\n1. Frequency distribution of surnames/initials: If many persons share the same surname, the corresponding blocks in s1 become very large, increasing C and reducing Save. For s2 this effect is even stronger, since fewer possible two-letter combinations exist, which means that common initials can cover thousands of persons, resulting in very large blocks with many unnecessary comparisons.\n\n2. Special Characters: Names containing Umlaute or special characters produce different block keys, even though they refer to the same person. s1 is directly affected by this, since even a minor difference in the surname spelling results in different blocks and the pair is permanently missed. s2 is less affected, as long as the first character of the surname remains the same, the pair still lands in the same block and is not lost. But if the first character is affected, s2 encounters the same problem. \n'